<a href="https://colab.research.google.com/github/rchirutkar/AgenticAI/blob/main/Week_7_Day_1_Learner_Planning_StructuredOutputs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 7 Day 1: Planning, Decisioning & Structured Outputs — Learner Notebook

        **Duration:** 150 minutes  
        **Continuum:** Week 7 sequential research agent → Week 8 parallel execution → Week 9 human review and resumable state

        ## By the end, you will be able to

- Explain why free-form LLM output breaks agent workflows.
- Define structured response schemas with Pydantic.
- Create a provider-neutral structured planning response using Groq/OpenRouter.
- Design a planning + decisioning flow for a domain research use case.


        Work through cells in order. Live LLM cells require a free Groq or OpenRouter key in Colab Secrets. Tavily search can run keylessly. Missing keys produce a clear setup error.


## Environment and API policy

- Start in **Google Colab**.
- Use **Groq** by default or **OpenRouter** as an alternative.
- Store keys in Colab Secrets; `.env` files are optional for local development and are not required here.
- Tavily Search runs keylessly for light classroom use; a free Tavily key is optional for higher limits.

> **Free-tier note:** Quotas and model capabilities can change. Groq is the recommended classroom default; with OpenRouter, set `OPENROUTER_MODEL` to a currently available model if the free router does not support the requested feature.

> The `openai` Python package is used as an OpenAI-compatible client for Groq/OpenRouter. The notebooks do not request an OpenAI API key.

In [ ]:
# Run once in a fresh Google Colab or Jupyter environment.
%pip install -q pydantic openai httpx langgraph langgraph-checkpoint-sqlite ipywidgets

from pathlib import Path
Path("artifacts").mkdir(exist_ok=True)
print("Setup complete. Next: add GROQ_API_KEY or OPENROUTER_API_KEY in Colab Secrets.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.4/163.4 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 60.1 MB/s eta 0:00:00
Setup complete. Next: add GROQ_API_KEY or OPENROUTER_API_KEY in Colab Secrets.


In [ ]:
# Select the provider. Groq is the default; OpenRouter is the alternative.
import os
os.environ.setdefault("LLM_PROVIDER", "groq")

# Optional model overrides:
# os.environ["GROQ_MODEL"] = "openai/gpt-oss-20b"
# os.environ["OPENROUTER_MODEL"] = "openrouter/free"
print("Selected provider:", os.environ["LLM_PROVIDER"])

Selected provider: groq


In [ ]:
import os
from typing import Optional


def load_secret(name: str, *, required: bool = False) -> Optional[str]:
    """Load a secret from the local environment or Google Colab Secrets."""
    value = os.getenv(name)
    if not value:
        try:
            from google.colab import userdata  # type: ignore
            value = userdata.get(name)
        except Exception:
            value = None
    if value:
        os.environ[name] = value
    if required and not value:
        raise RuntimeError(
            f"Missing {name}. In Colab, open the key icon (Secrets), add {name}, "
            "and enable notebook access. Locally, set it as an environment variable."
        )
    return value


LLM_PROVIDER = os.getenv("LLM_PROVIDER", "groq").strip().lower()
if LLM_PROVIDER not in {"groq", "openrouter"}:
    raise ValueError("LLM_PROVIDER must be 'groq' or 'openrouter'.")


def get_llm_client_and_model():
    """Return an OpenAI-compatible client configured for Groq or OpenRouter."""
    from openai import OpenAI

    if LLM_PROVIDER == "groq":
        api_key = load_secret("GROQ_API_KEY", required=True)
        model = os.getenv("GROQ_MODEL", "openai/gpt-oss-20b")
        client = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=api_key)
        return client, model

    api_key = load_secret("OPENROUTER_API_KEY", required=True)
    model = os.getenv("OPENROUTER_MODEL", "openrouter/free")
    client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=api_key,
        default_headers={"X-OpenRouter-Title": "Agentic AI Learning Lab"},
    )
    return client, model


configured_key = "GROQ_API_KEY" if LLM_PROVIDER == "groq" else "OPENROUTER_API_KEY"
print(f"LLM provider: {LLM_PROVIDER}")
print(f"Model override: {os.getenv('GROQ_MODEL') or os.getenv('OPENROUTER_MODEL') or 'not set; notebook default will be used'}")
print(f"{configured_key} available:", bool(load_secret(configured_key)))

LLM provider: groq
Model override: not set; notebook default will be used
GROQ_API_KEY available: True


In [ ]:
# Provider configuration is loaded in the previous cell.
print("Secrets are managed through Colab Secrets or local environment variables.")

Secrets are managed through Colab Secrets or local environment variables.


## Define data contracts


In [ ]:
from __future__ import annotations
from typing import Literal, Optional
from pydantic import BaseModel, Field

class SubQuestion(BaseModel):
    question: str = Field(description="A clear research sub-question.")
    depth: Literal["surface", "deep"] = Field(description="Search depth.")
    expected_source: str = Field(description="Likely source type.")
    success_criteria: str = Field(description="What evidence would make this answered?")

class ResearchPlan(BaseModel):
    topic: str
    original_question: str
    sub_questions: list[SubQuestion] = Field(min_length=2, max_length=7)
    stopping_criteria: str
    risks_or_assumptions: list[str] = Field(default_factory=list)

class Finding(BaseModel):
    sub_question: str
    title: str
    url: str
    content: str
    source_score: float = Field(ge=0, le=1)
    reason_accepted: str

class ReviewVerdict(BaseModel):
    needs_deeper: bool
    follow_up_query: Optional[str] = None
    reason: str

class ResearchReport(BaseModel):
    executive_summary: str
    key_findings: list[str]
    source_notes: list[str]
    gaps: list[str]
    confidence_score: float = Field(ge=0, le=1)

class ReportReview(BaseModel):
    score: int = Field(ge=1, le=5)
    passed: bool
    strengths: list[str]
    issues: list[str]
    suggested_next_queries: list[str] = Field(default_factory=list)


## Inspect the schema


In [ ]:
import json
print(json.dumps(ResearchPlan.model_json_schema(), indent=2))


{
  "$defs": {
    "SubQuestion": {
      "properties": {
        "question": {
          "description": "A clear research sub-question.",
          "title": "Question",
          "type": "string"
        },
        "depth": {
          "description": "Search depth.",
          "enum": [
            "surface",
            "deep"
          ],
          "title": "Depth",
          "type": "string"
        },
        "expected_source": {
          "description": "Likely source type.",
          "title": "Expected Source",
          "type": "string"
        },
        "success_criteria": {
          "description": "What evidence would make this answered?",
          "title": "Success Criteria",
          "type": "string"
        }
      },
      "required": [
        "question",
        "depth",
        "expected_source",
        "success_criteria"
      ],
      "title": "SubQuestion",
      "type": "object"
    }
  },
  "properties": {
    "topic": {
      "title": "Topic",
      "type":

## Make a real Groq/OpenRouter structured response call


In [ ]:
import json
from pydantic import ValidationError


def _json_schema_response_format(schema_model: type[BaseModel]) -> dict:
    return {
        "type": "json_schema",
        "json_schema": {
            "name": schema_model.__name__.lower(),
            "strict": False,
            "schema": schema_model.model_json_schema(),
        },
    }


def _chat_json(client, model: str, messages: list[dict], schema_model: type[BaseModel], temperature: float):
    """Try JSON Schema, then JSON object, then prompt-only JSON mode."""
    attempts = [
        {"response_format": _json_schema_response_format(schema_model), "label": "JSON Schema"},
        {"response_format": {"type": "json_object"}, "label": "JSON object"},
        {"label": "prompt-only JSON"},
    ]
    last_error = None
    for attempt in attempts:
        kwargs = {
            "model": model,
            "messages": messages,
            "temperature": temperature,
        }
        if "response_format" in attempt:
            kwargs["response_format"] = attempt["response_format"]
        try:
            return client.chat.completions.create(**kwargs), attempt["label"]
        except Exception as exc:
            last_error = exc
            print(f"{attempt['label']} mode unavailable: {type(exc).__name__}. Trying the next mode.")
    raise RuntimeError("The selected model could not return JSON in any supported mode.") from last_error


def llm_structured(prompt: str, schema_model: type[BaseModel], *, temperature: float = 0.2):
    """Call Groq/OpenRouter and validate the response with Pydantic."""
    client, model = get_llm_client_and_model()
    schema_text = json.dumps(schema_model.model_json_schema())
    messages = [
        {
            "role": "system",
            "content": "Return only one valid JSON object. Follow the supplied JSON Schema exactly.",
        },
        {"role": "user", "content": f"{prompt}\n\nJSON Schema:\n{schema_text}"},
    ]

    response, mode_used = _chat_json(client, model, messages, schema_model, temperature)
    print("Structured-output mode used:", mode_used)
    text = response.choices[0].message.content or "{}"
    try:
        return schema_model.model_validate_json(text)
    except ValidationError as validation_error:
        repair_prompt = f"""
The previous JSON failed validation.
Validation error:
{validation_error}

Previous JSON:
{text}

Return one corrected JSON object only, matching this schema:
{schema_text}
"""
        repaired, repair_mode = _chat_json(
            client,
            model,
            messages + [{"role": "user", "content": repair_prompt}],
            schema_model,
            0.0,
        )
        print("Repair mode used:", repair_mode)
        return schema_model.model_validate_json(repaired.choices[0].message.content or "{}")


In [ ]:
my_question = "How can an HR team use AI to reduce employee attrition while staying compliant with privacy rules?"
my_prompt = f"""
Create a research plan for this question:
{my_question}
Return 3 to 5 sub-questions with depth, expected source, success criteria, stopping criteria and risks.
"""
my_plan = llm_structured(my_prompt, ResearchPlan)
my_plan


Structured-output mode used: JSON Schema


ResearchPlan(topic='HR AI for Attrition Reduction and Privacy Compliance', original_question='How can an HR team use AI to reduce employee attrition while staying compliant with privacy rules?', sub_questions=[SubQuestion(question='What AI-driven predictive analytics methods can identify at‑risk employees without violating privacy regulations?', depth='deep', expected_source='academic journals or industry whitepapers', success_criteria='Evidence of models that use anonymized data and comply with GDPR/CCPA'), SubQuestion(question='How can AI-powered engagement tools be designed to respect employee consent and data minimization principles?', depth='surface', expected_source='HR tech blogs, case studies', success_criteria='Guidelines that incorporate opt‑in mechanisms and data minimization'), SubQuestion(question='What governance frameworks are needed to monitor AI decision‑making in HR to ensure transparency and accountability?', depth='deep', expected_source='regulatory guidance, compli

In [ ]:
import json
print(json.dumps(my_plan.model_dump(), indent=2))

{
  "topic": "HR AI for Attrition Reduction and Privacy Compliance",
  "original_question": "How can an HR team use AI to reduce employee attrition while staying compliant with privacy rules?",
  "sub_questions": [
    {
      "question": "What AI-driven predictive analytics methods can identify at\u2011risk employees without violating privacy regulations?",
      "depth": "deep",
      "expected_source": "academic journals or industry whitepapers",
      "success_criteria": "Evidence of models that use anonymized data and comply with GDPR/CCPA"
    },
    {
      "question": "How can AI-powered engagement tools be designed to respect employee consent and data minimization principles?",
      "depth": "surface",
      "expected_source": "HR tech blogs, case studies",
      "success_criteria": "Guidelines that incorporate opt\u2011in mechanisms and data minimization"
    },
    {
      "question": "What governance frameworks are needed to monitor AI decision\u2011making in HR to ensure 

In [ ]:
# Try By Yourself:

my_question = <your question here>


my_prompt = f"""
Create a research plan for this question:
{my_question}
Return 3 to 5 sub-questions with depth, expected source, success criteria, stopping criteria and risks.
"""
my_plan = llm_structured(my_prompt, ResearchPlan)


print(json.dumps(my_plan.model_dump(), indent=2))

## Reflection
Why is a structured research plan better than asking an LLM to 'just research this'?
